# 04 - Gold

Copyright (c) Microsoft Corporation. Licensed under the MIT License.

Builds the **gold-layer Delta tables** that back the Direct Lake *Fabric Arch Review - Governance* semantic model and report using sanitized architectural metadata and the versioned profile. Reads `findings.json` + `raw/` from the same `RUN_ID` folder and **replaces** that run's rows in the Lakehouse `Tables/`, preserving history while making retries idempotent. Owner projection is enabled only when optional base setup has bootstrapped `owner_access`; disabling the deployment flag later does not remove that table or stop projection. Owner tables contain a conservative technical subset, not mixed findings, raw expressions/source code or full FAR scores. Notebook, Dataflow and pipeline-dependency evidence requires explicit workspace/item provenance; unattributed or name-only rows are excluded, and all owner assessments remain incomplete.

Owner history starts with owner-enabled runs; older runs require explicit historical Gold reruns, not automatic backfill. A newly projected workspace has no access grant until the independent `07_OwnerAccessSync` completes. Run sync after the first review and before testing new workspaces, or wait for its daily run. Sync covers all owner-projected workspaces, not only this week's top five or email recipients; review completion never grants permissions. No Azure/Fabric admin auth is needed for review projection - this stage uses local artifacts and Spark; deployment-version update checks can contact GitHub.


## Parameters


In [ ]:
# Parameters (injected by the pipeline)
CLIENT_NAME = "Contoso"
ENGAGEMENT_NAME = "Fabric Architecture Review"
REVIEWER_NAME = ""
RUN_ID = "latest"
# Run configuration bindings populated by setup.
FAR_THRESHOLDS_LIBRARY = "FAR_Thresholds"
FAR_CONNECTIONS_LIBRARY = "FAR_Connections"


## Load staged FAR runtime


In [ ]:
import os, sys, runpy
from pathlib import Path
import notebookutils
ctx = notebookutils.runtime.context
if (ctx.get("defaultLakehouseName") != "FOS_Lakehouse" or not ctx.get("defaultLakehouseId")
        or ctx.get("defaultLakehouseWorkspaceId") != (ctx.get("currentWorkspaceId") or ctx.get("workspaceId"))):
    raise ValueError("Attach this workspace's FOS_Lakehouse as default and restart the session.")
REPO_DIR = "/lakehouse/default/Files/far/runtime/source"
if not Path(REPO_DIR, "fabric", "bootstrap.py").is_file():
    raise FileNotFoundError("FAR runtime not staged. Deploy the architecture-review module before running.")
runpy.run_path(str(Path(REPO_DIR, "fabric", "bootstrap.py")))


## Build Gold tables: replace this run and preserve other history


In [ ]:
from orchestration.storage import far_table
import os, datetime as dt
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (StructType, StructField, StringType, LongType,
                               DoubleType, BooleanType, TimestampType)
from delta.tables import DeltaTable
from reports.gold_layer import build_gold_from_dir
from reports.powerbi.schema import GOLD_TABLES_BY_NAME, coerce_value

from orchestration.storage import latest_run_id, run_folder
if RUN_ID == "latest":
    RUN_ID = latest_run_id()
OUT_DIR = str(run_folder(RUN_ID))
from pathlib import Path
from orchestration.review_config import activate
activate(Path(OUT_DIR) / "review_config.json")
import notebookutils
from orchestration.review_workspace import export_review_workspace
export_review_workspace(notebookutils)
if not os.path.isdir(os.path.join(OUT_DIR, "raw")):
    raise RuntimeError("Run folder not found: " + OUT_DIR + " -- run Collect + Analyze first.")

RUN_TS = dt.datetime.now(dt.timezone.utc).replace(microsecond=0, tzinfo=None).isoformat() + "Z"

spark = SparkSession.builder.getOrCreate()
# Read the installed version locally; never check for new source or releases online.
_deployed_version = None
try:
    _mrows = spark.table(far_table("meta_deployment")).select("version").collect()
    if _mrows:
        _deployed_version = _mrows[0]["version"]
        print("  deployed FAR version:", _deployed_version)
except Exception as _e:
    print("  meta_deployment not found (using code VERSION):", _e)
tables = build_gold_from_dir(
    OUT_DIR, run_id=RUN_ID, run_timestamp=RUN_TS,
    client_name=CLIENT_NAME, engagement_name=ENGAGEMENT_NAME, reviewer_name=REVIEWER_NAME,
    deployed_version=_deployed_version, repo_url="", branch="", check_remote=False,
)

_TYPE = {"string": StringType(), "int64": LongType(), "double": DoubleType(),
         "boolean": BooleanType(), "dateTime": TimestampType()}

def _schema(table):
    return StructType([StructField(c.name, _TYPE[c.kind], True) for c in table.columns])

def _rows(rows, table):
    cols = [(c.name, c.kind) for c in table.columns]
    out = []
    for r in rows:
        rec = []
        for name, kind in cols:
            v = r.get(name)
            if kind == "dateTime" and isinstance(v, str) and v:
                v = dt.datetime.fromisoformat(v.replace("Z", "+00:00")).replace(tzinfo=None)
            rec.append(coerce_value(kind, v, table.name + "." + name))
        out.append(tuple(rec))
    return out

# Materialize every DataFrame before changing any Delta table so schema or row
# conversion failures cannot leave a partially replaced run.
frames = {}
for name, rows in tables.items():
    table = GOLD_TABLES_BY_NAME[name]
    if "run_id" not in {column.name for column in table.columns}:
        raise RuntimeError("Gold table lacks run_id replacement key: " + name)
    frames[name] = spark.createDataFrame(_rows(rows, table), schema=_schema(table))

print("Gold build for run", RUN_ID, "@", RUN_TS)
for name, df in frames.items():
    if spark.catalog.tableExists(far_table(name)):
        DeltaTable.forName(spark, far_table(name)).delete(F.col("run_id") == F.lit(RUN_ID))
    (df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(far_table(name)))
    expected_count = len(tables[name])
    actual_count = spark.table(far_table(name)).where(F.col("run_id") == RUN_ID).count()
    if actual_count != expected_count:
        raise RuntimeError(name + " verification failed: expected " + str(expected_count) +
                           " row(s), found " + str(actual_count))
    print("  " + name + ": replaced run with " + str(actual_count) + " row(s)")

# Flag only the newest run as is_latest so the Direct Lake report defaults to it.
_max_ts = spark.table(far_table("gold_run_summary")).agg(F.max("run_timestamp").alias("m")).collect()[0]["m"]
DeltaTable.forName(spark, far_table("gold_run_summary")).update(
    set={"is_latest": F.col("run_timestamp") == F.lit(_max_ts)}
)
print("  gold_run_summary: is_latest set for newest run @", _max_ts)

from reports.owner.fabric_runtime import materialize_owner_gold
materialize_owner_gold(spark, tables)

from reports.far_evidence import persist
persist(spark, Path(OUT_DIR), RUN_ID)

try:
    import notebookutils
    notebookutils.notebook.exit("gold:" + OUT_DIR)
except Exception:
    pass
